# TP — Comptage de mots et comparaison des méthodes

**Nom :** Alex Abdallah  
**Cours :** Fondements et Applications de Big Data

## Objectif

Comparer trois méthodes de comptage des occurrences de mots :
1. Python séquentiel simple.
2. Pseudo-MapReduce en Python.
3. Hadoop MapReduce.

Les trois méthodes seront testées sur les mêmes fichiers,
avec des tailles croissantes.

In [1]:
from pathlib import Path
import random

dossier = Path("data")
dossier.mkdir(exist_ok=True)

N = 1_000
facteurs = [1, 10, 100, 1_000]

vocabulaire = [
    "chat", "chien", "arbre", "fleur", "maison",
    "soleil", "montagne", "livre", "ciel", "mer"
]

# Même fichier initial à chaque exécution.
generateur = random.Random(42)
lignes = [generateur.choice(vocabulaire) + "\n" for _ in range(N)]
bloc_initial = "".join(lignes)

fichiers = {}

for m in facteurs:
    chemin = dossier / f"mots_m{m}.txt"

    with chemin.open("w", encoding="utf-8") as fichier:
        for _ in range(m):
            fichier.write(bloc_initial)

    fichiers[m] = chemin

    print(
        f"{chemin.name} : {N * m:,} lignes, "
        f"{chemin.stat().st_size:,} octets"
    )

mots_m1.txt : 1,000 lignes, 7,117 octets
mots_m10.txt : 10,000 lignes, 71,170 octets
mots_m100.txt : 100,000 lignes, 711,700 octets
mots_m1000.txt : 1,000,000 lignes, 7,117,000 octets


## Méthode 1 — Python simple

Le programme lit le fichier et utilise un dictionnaire.
À chaque mot rencontré, son compteur augmente de 1.

In [2]:
def compter_python(chemin):
    with open(chemin, "r", encoding="utf-8") as fichier:
        mots = fichier.read().split()

    compteur = {}

    for mot in mots:
        compteur[mot] = compteur.get(mot, 0) + 1

    return compteur


resultat = compter_python(fichiers[1])
print(resultat)
print("Total :", sum(resultat.values()))

{'chien': 108, 'chat': 91, 'maison': 113, 'fleur': 116, 'arbre': 85, 'ciel': 103, 'mer': 95, 'montagne': 102, 'livre': 105, 'soleil': 82}
Total : 1000


## Méthode 2 — Pseudo-MapReduce Python

- Map : transforme chaque mot en une paire (mot, 1).
- Shuffle : regroupe les valeurs associées au même mot.
- Reduce : additionne ces valeurs.

Cette version s'exécute localement, sans parallélisation.

In [3]:
from collections import defaultdict


def mapper(ligne):
    return [(mot, 1) for mot in ligne.strip().split()]


def shuffle(paires):
    groupes = defaultdict(list)

    for mot, valeur in paires:
        groupes[mot].append(valeur)

    return groupes


def reducer(groupes):
    return {
        mot: sum(valeurs)
        for mot, valeurs in groupes.items()
    }


def compter_mapreduce(chemin):
    paires = []

    with open(chemin, "r", encoding="utf-8") as fichier:
        for ligne in fichier:
            paires.extend(mapper(ligne))

    return reducer(shuffle(paires))


resultat_mr = compter_mapreduce(fichiers[1])

assert resultat_mr == resultat
print("Les deux méthodes donnent les mêmes résultats.")

Les deux méthodes donnent les mêmes résultats.


In [4]:
from time import perf_counter
from statistics import median

mesures = []
reference = compter_python(fichiers[1])

methodes = {
    "Python simple": compter_python,
    "Pseudo-MapReduce": compter_mapreduce,
}

for m, chemin in fichiers.items():
    attendu = {
        mot: nombre * m
        for mot, nombre in reference.items()
    }

    for nom, fonction in methodes.items():
        durees = []

        for _ in range(3):
            debut = perf_counter()
            comptes = fonction(chemin)
            durees.append(perf_counter() - debut)

            assert comptes == attendu

        temps = median(durees)

        mesures.append({
            "m": m,
            "lignes": N * m,
            "octets": chemin.stat().st_size,
            "methode": nom,
            "temps_s": temps,
        })

        print(f"{N * m:>10,} lignes | {nom:<18} | {temps:.6f} s")

     1,000 lignes | Python simple      | 0.000633 s
     1,000 lignes | Pseudo-MapReduce   | 0.001514 s
    10,000 lignes | Python simple      | 0.003527 s
    10,000 lignes | Pseudo-MapReduce   | 0.008591 s
   100,000 lignes | Python simple      | 0.025055 s
   100,000 lignes | Pseudo-MapReduce   | 0.081694 s
 1,000,000 lignes | Python simple      | 0.271078 s
 1,000,000 lignes | Pseudo-MapReduce   | 0.925121 s


## Méthode 3 — Hadoop MapReduce avec Docker Compose

Hadoop est exécuté dans quatre conteneurs :

- NameNode : gère les métadonnées de HDFS.
- DataNode : stocke les blocs de données.
- ResourceManager : gère les ressources avec YARN.
- NodeManager : lance les tâches sur le nœud de calcul.

Les fichiers sont copiés dans HDFS avant les tests.
Le programme WordCount fourni avec Hadoop est utilisé
pour compter les occurrences des mots.

Tous les conteneurs fonctionnent sur un seul ordinateur.
Il s'agit donc d'un environnement Hadoop local à plusieurs
services, et non d'un cluster de plusieurs machines physiques.

### Protocole de mesure

Chaque fichier est testé trois fois.
La médiane des trois durées est retenue.

La durée mesurée inclut la commande Docker, la soumission
du job Hadoop et son exécution.

Elle exclut le téléchargement de l'image, le démarrage
des services, le transfert initial dans HDFS et la lecture
finale des résultats.

Les comptes sont vérifiés pour chaque mot. Pour un fichier
répété m fois, chaque compteur doit être multiplié par m.

In [5]:
%pip install pandas matplotlib

Note: you may need to restart the kernel to use updated packages.


## Méthode 3 — Hadoop MapReduce

Hadoop est exécuté avec Docker Compose. Les fichiers de test
sont stockés dans HDFS et le programme WordCount fourni
avec Hadoop est exécuté avec YARN.

Map produit une paire (mot, 1) pour chaque occurrence.
Shuffle regroupe les valeurs par mot.
Reduce additionne les valeurs pour obtenir le nombre
d'occurrences de chaque mot.

Chaque fichier est testé et son temps d'exécution est mesuré.
Les résultats sont comparés à ceux de Python simple.

Le temps mesuré inclut le lancement de la commande Docker
et l'exécution du job Hadoop. Le transfert préalable dans
HDFS et la récupération du résultat sont exclus.

In [8]:
import subprocess
import time
from pathlib import Path

dossier_tp = Path.cwd()
mesures_hadoop = []

def executer_hadoop(*arguments):
    return subprocess.check_output(
        [
            "docker", "compose", "exec", "-T",
            "namenode", *arguments
        ],
        cwd=dossier_tp,
        text=True,
        encoding="utf-8",
        errors="replace"
    )

print(executer_hadoop("hadoop", "version"))

# Un nom différent à chaque lancement évite les dossiers
# de sortie déjà existants dans HDFS.
identifiant = time.time_ns()

for m, chemin in fichiers.items():
    entree = f"/tp/input/mots_m{m}.txt"
    sortie = f"/tp/resultat_{identifiant}_m{m}"

    commande = (
        "hadoop jar "
        "/opt/hadoop/share/hadoop/mapreduce/"
        "hadoop-mapreduce-examples-*.jar "
        f"wordcount {entree} {sortie}"
    )

    debut = time.perf_counter()

    execution = subprocess.run(
        [
            "docker", "compose", "exec", "-T",
            "namenode", "bash", "-lc", commande
        ],
        cwd=dossier_tp,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace"
    )

    duree = time.perf_counter() - debut

    if execution.returncode != 0:
        print(execution.stdout)
        print(execution.stderr)
        raise RuntimeError("Le job Hadoop a échoué.")

    texte = executer_hadoop(
        "hdfs", "dfs", "-cat", f"{sortie}/part-r-*"
    )

    comptes = {}
    for ligne in texte.splitlines():
        mot, nombre = ligne.split("\t")
        comptes[mot] = int(nombre)

    assert comptes == compter_python(chemin), (
        f"Résultats différents pour {chemin.name}"
    )

    mesures_hadoop.append({
        "m": m,
        "lignes": N * m,
        "methode": "Hadoop",
        "temps_s": duree
    })

    print(f"\nFichier : {chemin.name}")
    print(f"Nombre de lignes : {N * m:,}")
    print(f"Temps Hadoop : {duree:.6f} s")
    print("Résultats identiques à Python simple.")

    for mot, nombre in sorted(
        comptes.items(),
        key=lambda element: element[1],
        reverse=True
    ):
        print(f"{mot} : {nombre}")

Hadoop 3.3.6
Source code repository https://github.com/apache/hadoop.git -r 1be78238728da9266a4f88195058f08fd012bf9c
Compiled by ubuntu on 2023-06-18T08:22Z
Compiled on platform linux-x86_64
Compiled with protoc 3.7.1
From source with checksum 5652179ad55f76cb287d9c633bb53bbd
This command was run using /opt/hadoop/share/hadoop/common/hadoop-common-3.3.6.jar


Fichier : mots_m1.txt
Nombre de lignes : 1,000
Temps Hadoop : 27.716537 s
Résultats identiques à Python simple.
fleur : 116
maison : 113
chien : 108
livre : 105
ciel : 103
montagne : 102
mer : 95
chat : 91
arbre : 85
soleil : 82

Fichier : mots_m10.txt
Nombre de lignes : 10,000
Temps Hadoop : 26.657770 s
Résultats identiques à Python simple.
fleur : 1160
maison : 1130
chien : 1080
livre : 1050
ciel : 1030
montagne : 1020
mer : 950
chat : 910
arbre : 850
soleil : 820

Fichier : mots_m100.txt
Nombre de lignes : 100,000
Temps Hadoop : 26.766399 s
Résultats identiques à Python simple.
fleur : 11600
maison : 11300
chien : 10800
livre :

In [10]:
import pandas as pd

python_df = pd.DataFrame(mesures)
hadoop_df = pd.DataFrame(mesures_hadoop)

ensemble = pd.concat(
    [
        python_df[["lignes", "methode", "temps_s"]],
        hadoop_df[["lignes", "methode", "temps_s"]]
    ],
    ignore_index=True
)

comparaison = ensemble.pivot(
    index="lignes",
    columns="methode",
    values="temps_s"
).sort_index()

comparaison = comparaison[
    ["Python simple", "Pseudo-MapReduce", "Hadoop"]
]

comparaison.columns = [
    "Python simple (s)",
    "Pseudo-MapReduce (s)",
    "Hadoop (s)"
]

comparaison.index.name = "Nombre de lignes"

display(comparaison.round(6))

,Python simple (s),Pseudo-MapReduce (s),Hadoop (s)
Nombre de lignes,,,
1000,0.000633,0.001514,27.716537
10000,0.003527,0.008591,26.657770
100000,0.025055,0.081694,26.766399
1000000,0.271078,0.925121,27.393482


## Comparaison et analyse

Les trois méthodes ont été testées sur les mêmes fichiers,
contenant 1 000, 10 000, 100 000 et 1 000 000 de lignes.

Les résultats de comptage sont identiques pour les trois
méthodes.

Les temps Python correspondent à la médiane de trois
exécutions. Le temps Hadoop correspond à une exécution
par fichier.

Pour 1 000 lignes, [méthode] est la plus rapide,
avec un temps de [temps] seconde(s).

Pour 1 000 000 de lignes, [méthode] est la plus rapide,
avec un temps de [temps] seconde(s).

Python simple compte directement les mots dans un dictionnaire.
Le pseudo-MapReduce ajoute la construction des paires et
le regroupement des valeurs avant leur addition.

Hadoop ajoute une surcharge liée à la soumission du job
et au lancement des tâches. Cette surcharge peut expliquer
ses temps plus élevés sur les fichiers testés.

## Conclusion

Ce TP a permis d'implémenter et de tester les trois approches
de comptage de mots, puis de comparer leurs temps sur
des fichiers de tailles croissantes.

Sur les tailles étudiées, [méthode] est la plus rapide.

Hadoop fonctionne ici dans des conteneurs sur un seul
ordinateur. Les mesures ne permettent donc pas d'évaluer
les performances d'un cluster de plusieurs machines.
De plus, le temps Hadoop inclut le lancement de la commande,
alors que le temps Python exclut le démarrage de l'interpréteur.